# 6.1. 从全连接层到卷积

到目前为止，我们的模型都是由全连接层组成的。然而，当处理图像数据时，全连接层存在一些问题：
1. **参数过多**：对于高分辨率图像，全连接层需要大量参数
2. **忽略空间结构**：将图像展平会丢失空间信息

卷积神经网络通过以下两个原则解决这些问题：
1. **平移不变性（translation invariance）**：不管检测对象出现在图像中的哪个位置，神经网络都应该对相同的图像区域产生相似的响应
2. **局部性（locality）**：神经网络的前面几层应该只探索输入图像中的局部区域，而不过度考虑图像中相距较远区域的关系

In [ ]:
import tensorflow as tf
import numpy as np

## 6.1.1. 全连接层的问题

假设我们有一个输入图像（高度和宽度各为1000像素），隐藏层也有1000个神经元。

In [ ]:
# 全连接层的参数数量
input_size = 1000 * 1000  # 1000x1000 图像
hidden_size = 1000
num_params = input_size * hidden_size
print(f"全连接层参数数量: {num_params:,}")
print(f"这相当于 {num_params / 1e9:.1f} 十亿个参数！")

## 6.1.2. 卷积层的优势

卷积层通过以下方式减少参数：
1. 使用小的滤波器（kernel）而不是全连接权重
2. 在整个图像上共享这些滤波器
3. 只关注局部区域

In [ ]:
# 卷积层的参数数量
kernel_size = 5  # 5x5 卷积核
num_filters = 64  # 64个滤波器
conv_params = kernel_size * kernel_size * num_filters
print(f"卷积层参数数量: {conv_params:,}")
print(f"参数减少了 {num_params / conv_params:,.0f} 倍！")

## 6.1.3. 平移不变性

对于图像 $X$，无论目标出现在哪里，卷积层都应该给出相似的响应。

In [ ]:
# 演示平移不变性
# 创建一个简单的图像，中间有一个亮点
def create_image_with_point(position):
    img = np.zeros((8, 8))
    img[position[0], position[1]] = 1.0
    return img

# 创建不同位置的图像
img1 = create_image_with_point((3, 3))
img2 = create_image_with_point((3, 5))

print("图像1 (亮点在中间):")
print(img1)
print("\n图像2 (亮点在右边):")
print(img2)

## 6.1.4. 局部性

卷积层只关注输入的局部区域，而不是整个输入。

In [ ]:
# 展示局部性：3x3 卷积核只看局部区域
def show_receptive_field(img_size=8, kernel_size=3):
    img = np.zeros((img_size, img_size))
    # 标记感受野
    start = img_size // 2 - kernel_size // 2
    end = start + kernel_size
    img[start:end, start:end] = 1.0
    return img

receptive_field = show_receptive_field()
print("3x3 卷积核的局部感受野（1表示关注的区域）:")
print(receptive_field)

## 小结

1. **平移不变性**：卷积层对图像中相同的模式产生相似的响应，无论它出现在哪里
2. **局部性**：卷积层只关注局部区域，大大减少了参数数量
3. **参数共享**：同一个卷积核在整个图像上滑动，共享参数
4. **空间结构保留**：与全连接层不同，卷积层保留了输入的空间结构